# 03. Data Mining: Association Rules & Clustering

## Superstore Sales Data Mining Project

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder
from src.mining.association import AssociationMiner
from src.mining.clustering import ClusterMiner
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load & Prepare Data

In [3]:
# Load data
loader = DataLoader()
df = loader.generate_sample_data(n_orders=2000)

# Clean data
cleaner = DataCleaner(df)
df = cleaner.handle_missing_values()
df, _ = cleaner.handle_duplicates()

# Create features
builder = FeatureBuilder(df)
rfm = builder.create_rfm_features()
basket = builder.create_basket_data(min_items=2)

print(f"RFM shape: {rfm.shape}")
print(f"Basket transactions: {len(basket)}")

INFO:src.data.loader:Generated 4982 sample records
INFO:src.data.cleaner:Handling missing values...
INFO:src.data.cleaner:Handling duplicates...
INFO:src.data.cleaner:Removed 0 duplicate rows
INFO:src.features.builder:Creating RFM features...


TypeError: unsupported operand type(s) for +: 'int' and 'datetime.timedelta'

## 2. Association Rule Mining

In [ ]:
# Prepare transactions
transactions = basket['Items'].tolist()

# Run Apriori
miner = AssociationMiner(min_support=0.02, min_confidence=0.3, min_lift=1.0)
result = miner.fit(transactions)

# Top rules
print("Top 10 Association Rules:")
top_rules = miner.get_top_rules(n=10, sort_by='lift')
print(top_rules)

In [ ]:
# Metrics
metrics = miner.get_rule_metrics()
print("\nRule Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Insights
print("\nBusiness Insights from Association Rules:")
for insight in miner.generate_insights():
    print(f"  - {insight}")

## 3. Clustering Analysis

In [ ]:
# Prepare RFM for clustering
feature_cols = ['Recency', 'Frequency', 'Monetary']
scaler = StandardScaler()
features = scaler.fit_transform(rfm[feature_cols])

# Find optimal K
clusterer = ClusterMiner(n_clusters=4, random_state=42)
k_results = clusterer.find_optimal_k(features, range(2, 8))

# Plot elbow curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(k_results['k'], k_results['silhouette'], 'bo-')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_results['k'], k_results['davies_bouldin'], 'ro-')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Davies-Bouldin Index')
axes[1].set_title('DBI (lower is better)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(k_results['k'], k_results['calinski_harabasz'], 'go-')
axes[2].set_xlabel('Number of Clusters (K)')
axes[2].set_ylabel('Calinski-Harabasz Index')
axes[2].set_title('CH (higher is better)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/07_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: outputs/figures/07_optimal_k.png")

In [ ]:
# Fit K-Means with optimal K (let's use 4)
optimal_k = 4
labels = clusterer.fit_kmeans(features, n_clusters=optimal_k)

# Add labels to RFM
rfm['Cluster'] = labels

# Create profiles
profiles = clusterer.create_cluster_profiles(features, feature_cols, rfm)
print("\nCluster Profiles:")
print(profiles)

In [ ]:
# Cluster descriptions
print("\nCluster Descriptions:")
descriptions = clusterer.get_cluster_descriptions()
for cid, desc in descriptions.items():
    print(f"  {desc}")

In [ ]:
# Evaluate clustering
eval_metrics = clusterer.evaluate_clustering(features)
print("\nClustering Evaluation Metrics:")
for key, value in eval_metrics.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## 4. Visualize Clusters

In [ ]:
# Plot clusters
clusterer.visualize_clusters(features, method='pca', 
                           save_path='../outputs/figures/08_clusters_pca.png')
print("Saved: outputs/figures/08_clusters_pca.png")

In [ ]:
# Plot cluster profiles
fig, ax = plt.subplots(figsize=(12, 6))

profile_features = profiles[feature_cols].copy()
for col in feature_cols:
    max_val = profile_features[col].max()
    if max_val > 0:
        profile_features[col] = profile_features[col] / max_val

profile_features['Cluster'] = profiles['Cluster']
profile_features.set_index('Cluster').plot(kind='bar', ax=ax, width=0.8)

ax.set_xlabel('Cluster')
ax.set_ylabel('Normalized Value')
ax.set_title('Cluster Profiles (Normalized)')
ax.legend(title='Features')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('../outputs/figures/09_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: outputs/figures/09_cluster_profiles.png")

## 5. Save Results

In [ ]:
# Save results
top_rules.to_csv('../outputs/tables/03_association_rules.csv', index=False)
profiles.to_csv('../outputs/tables/03_cluster_profiles.csv', index=False)
rfm.to_csv('../data/processed/03_rfm_with_clusters.csv', index=False)

print("Data Mining completed!")
print("Saved:")
print("  - outputs/tables/03_association_rules.csv")
print("  - outputs/tables/03_cluster_profiles.csv")
print("  - data/processed/03_rfm_with_clusters.csv")